In [25]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/train-csv/treino_2.csv
/kaggle/input/test-csv/teste_2.csv


In [26]:
train = pd.read_csv("/kaggle/input/train-csv/treino_2.csv")
test = pd.read_csv("/kaggle/input/test-csv/teste_2.csv")

train.shape, test.shape

((90000, 6), (20513, 6))

In [27]:
train.sample(4)

,title,text,date,category,subcategory,link
15621,Blog da Seleção: Áudio de Rafinha explica um p...,De São Paulo - O áudio vazado de Rafinha criti...,2015-03-17,esporte,NaN,http://www1.folha.uol.com.br/esporte/2015/03/1...
39793,"Com caxumba, Neymar desfalca o Barcelona em de...",Diagnosticado com parotidite –infecção na glân...,2015-09-08,esporte,NaN,http://www1.folha.uol.com.br/esporte/2015/08/1...
14523,Virtudes do contorcionismo,O governo acaba de fazer um gesto diplomático ...,2015-04-15,colunas,matiasspektor,http://www1.folha.uol.com.br/colunas/matiasspe...
9337,Eleição na Holanda gera ansiedade entre imigra...,"A tulipa é uma flor imigrante. Seus bulbos, qu...",2017-03-15,mundo,NaN,http://www1.folha.uol.com.br/mundo/2017/03/186...


In [28]:
pip install -U spacy

Note: you may need to restart the kernel to use updated packages.


In [29]:
!python -m spacy download pt_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 22.6 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [30]:
import spacy
from time import time

nlp = spacy.load('pt_core_news_sm')
nlp

# Pré-processamento dos dados

In [31]:
text_titles = (titulos.lower() for titulos in train['title'])

In [32]:
def trata_texts(doc):
    tokens_validos = []
    for token in doc:
        valido = not token.is_stop and token.is_alpha
        if valido:
            tokens_validos.append(token.text)

    if len(tokens_validos) > 2:
        return " ".join(tokens_validos)

In [33]:
t0 = time()

texto_tratado = [trata_texts(doc) for doc in nlp.pipe(text_titles,
                                                     batch_size = 2000,
                                                     n_process = 12)]

#n_process = -1 -> utiliza todos núcleos do processador

t1 = time() - t0
print(t1/60)

3.2391515533129374


In [34]:
titulos_tratados = pd.DataFrame({'titulo': texto_tratado})
titulos_tratados

,titulo
0,polêmica marine le pen abomina negacionistas h...
1,macron le pen turno frança revés siglas tradic...
2,apesar larga vitória legislativas macron terá ...
3,governo antecipa balanço alckmin anuncia queda...
4,queda maio atividade econômica sobe junho bc
...,...
89995,mural há anos aeroporto recebido moradores gua...
89996,notícias schumacher boas ferrari
89997,olho bilhões governo conceder áreas petróleo
89998,moro deu lula papel coitadinho


In [35]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(sg = 0,
                    window = 2,
                    vector_size = 300,
                    min_count = 5,
                    alpha = 0.03,
                    min_alpha = 0.007)

'''
sg=1 -> skip=gram
sg=0 -> cbow

window -> quantidade de palavras antes e depois da 'target'

size -> tamanho do vetor

min_count -> quantidade mínima que a palavra precisa aparecer.
Bom para desconsiderar erros de dígitos

alpha -> learning_rate
'''


"\nsg=1 -> skip=gram\nsg=0 -> cbow\n\nwindow -> quantidade de palavras antes e depois da 'target'\n\nsize -> tamanho do vetor\n\nmin_count -> quantidade mínima que a palavra precisa aparecer.\nBom para desconsiderar erros de dígitos\n\nalpha -> learning_rate\n"

Skip-gram -> à partir da palavra, descobre o contexto

CBOW - > à partir do contexto, descobre a palavra

In [36]:
titulos_tratados.isna().sum()

titulo    5320
dtype: int64

In [37]:
titulos_tratados = titulos_tratados.dropna().drop_duplicates()
len(titulos_tratados)

84466

## Verificar logs do Word2Vec

In [43]:
import logging

lista_lista_tokens = [titulo.split(" ") for titulo in titulos_tratados.titulo]

logging.basicConfig(format='%(asctime)s: - %(message)s', level = logging.INFO)

w2v_model = Word2Vec(sg = 0,
                    window = 2,
                    vector_size = 300,
                    min_count = 5,
                    alpha = 0.03,
                    min_alpha = 0.007)

w2v_model.build_vocab(lista_lista_tokens, progress_per=5000)

# Treinamento do modelo CBOW

In [44]:
dir(w2v_model)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_adapt_by_suffix',
 '_check_corpus_sanity',
 '_check_training_sanity',
 '_clear_post_train',
 '_do_train_epoch',
 '_do_train_job',
 '_get_next_alpha',
 '_get_thread_working_mem',
 '_job_producer',
 '_load_specials',
 '_log_epoch_end',
 '_log_epoch_progress',
 '_log_progress',
 '_log_train_end',
 '_raw_word_count',
 '_save_specials',
 '_scan_vocab',
 '_smart_save',
 '_train_epoch',
 '_train_epoch_corpusfile',
 '_worker_loop',
 '_worker_loop_corpusfile',
 'add_lifecycle_event',
 'add_null_word',
 'alpha',
 'batch_words',
 'build_vocab',
 'build_vocab_from_freq',
 'cbow_mean',
 'comment',
 'compute_loss',
 'corpus_count',
 

In [45]:
w2v_model.corpus_count

84466

In [54]:
from gensim.models.callbacks import CallbackAny2Vec

# iniciando a chamada callback
class callback(CallbackAny2Vec):
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print('Loss após a época {}: {}'.format(self.epoch, loss))
        else:
            print('Loss após a época {}: {}'.format(self.epoch, 
                                                    loss - self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

In [55]:
w2v_model.train(lista_lista_tokens, 
               total_examples = w2v_model.corpus_count,
               epochs = 30,
               compute_loss = True,
               callbacks=[callback()])

Loss após a época 0: 150843.953125
Loss após a época 1: 145344.078125
Loss após a época 2: 139688.65625
Loss após a época 3: 126985.3125
Loss após a época 4: 121081.125
Loss após a época 5: 121997.0
Loss após a época 6: 111534.0625
Loss após a época 7: 114704.0625
Loss após a época 8: 102394.625
Loss após a época 9: 104229.5
Loss após a época 10: 101376.375
Loss após a época 11: 99116.25
Loss após a época 12: 86016.0
Loss após a época 13: 88439.75
Loss após a época 14: 91703.75
Loss após a época 15: 89630.375
Loss após a época 16: 88029.25
Loss após a época 17: 82376.0
Loss após a época 18: 80516.125
Loss após a época 19: 73307.5
Loss após a época 20: 68766.25
Loss após a época 21: 75763.75
Loss após a época 22: 74998.5
Loss após a época 23: 66271.75
Loss após a época 24: 68632.0
Loss após a época 25: 67362.75
Loss após a época 26: 62666.75
Loss após a época 27: 64916.25
Loss após a época 28: 65201.0
Loss após a época 29: 67377.5


(14583515, 16207260)

Uma das formas de avaliar é verificando a analogia entre as palavras

In [49]:
w2v_model.wv.most_similar('google')

[('apple', 0.5713045001029968),
 ('facebook', 0.5026540756225586),
 ('amazon', 0.474339097738266),
 ('airbnb', 0.4694391191005707),
 ('uber', 0.45809805393218994),
 ('volkswagen', 0.4508282244205475),
 ('tesla', 0.44974735379219055),
 ('software', 0.4429831802845001),
 ('yahoo', 0.44216644763946533),
 ('walmart', 0.4399779736995697)]

In [50]:
w2v_model.wv.most_similar('barça')

[('psg', 0.5900945067405701),
 ('madrid', 0.5842360258102417),
 ('lazio', 0.577834963798523),
 ('ancelotti', 0.5644336342811584),
 ('zidane', 0.5615102648735046),
 ('barcelona', 0.5592644214630127),
 ('parma', 0.5542382001876831),
 ('favoritismo', 0.5342880487442017),
 ('bayern', 0.5330793857574463),
 ('atlético', 0.5287938714027405)]

In [51]:
w2v_model.wv.most_similar('gm')

[('embraer', 0.6734641790390015),
 ('volks', 0.6573156714439392),
 ('chrysler', 0.6452614665031433),
 ('honda', 0.6368763446807861),
 ('braskem', 0.5941962003707886),
 ('volkswagen', 0.592688798904419),
 ('tesla', 0.5918386578559875),
 ('renault', 0.5793570876121521),
 ('inbev', 0.576449990272522),
 ('toyota', 0.5638071894645691)]

# Treinamento do modelo Skip-Gram

In [56]:
w2v_model_sg = Word2Vec(sg = 1,
                    window = 2,
                    vector_size = 300,
                    min_count = 5,
                    alpha = 0.03,
                    min_alpha = 0.007)

w2v_model_sg.build_vocab(lista_lista_tokens, progress_per=5000)

w2v_model_sg.train(lista_lista_tokens, 
               total_examples = w2v_model_sg.corpus_count,
               epochs = 30,
               compute_loss = True,
               callbacks=[callback()])

Loss após a época 0: 1331023.625
Loss após a época 1: 920678.125
Loss após a época 2: 835081.0
Loss após a época 3: 769720.0
Loss após a época 4: 673105.75
Loss após a época 5: 705395.5
Loss após a época 6: 612451.5
Loss após a época 7: 585388.0
Loss após a época 8: 529859.0
Loss após a época 9: 510934.0
Loss após a época 10: 491734.5
Loss após a época 11: 493220.0
Loss após a época 12: 502094.0
Loss após a época 13: 393321.0
Loss após a época 14: 425325.0
Loss após a época 15: 368921.0
Loss após a época 16: 357266.0
Loss após a época 17: 408826.0
Loss após a época 18: 337248.0
Loss após a época 19: 329459.0
Loss após a época 20: 338983.0
Loss após a época 21: 348430.0
Loss após a época 22: 324689.0
Loss após a época 23: 301127.0
Loss após a época 24: 327886.0
Loss após a época 25: 288328.0
Loss após a época 26: 315481.0
Loss após a época 27: 275834.0
Loss após a época 28: 274255.0
Loss após a época 29: 269269.0


(14584697, 16207260)

In [60]:
w2v_model_sg.wv.most_similar('barça')

[('ancelotti', 0.5876650214195251),
 ('zidane', 0.5623340606689453),
 ('espanyol', 0.5393019914627075),
 ('athletic', 0.5252688527107239),
 ('villarreal', 0.5224155783653259),
 ('betis', 0.5156114101409912),
 ('bilbao', 0.5139042139053345),
 ('cavani', 0.5091181397438049),
 ('madrid', 0.49851807951927185),
 ('celta', 0.4981610178947449)]

In [61]:
w2v_model_sg.wv.most_similar('google')

[('waze', 0.4468362331390381),
 ('reguladores', 0.42945390939712524),
 ('toshiba', 0.42072418332099915),
 ('walmart', 0.40846875309944153),
 ('snapchat', 0.39988014101982117),
 ('apple', 0.39958566427230835),
 ('chrysler', 0.38604480028152466),
 ('facebook', 0.38410109281539917),
 ('verizon', 0.38087454438209534),
 ('concorda', 0.3760179579257965)]

In [62]:
w2v_model_sg.wv.most_similar('gm')

[('volks', 0.6105590462684631),
 ('chrysler', 0.5925487875938416),
 ('metalúrgicos', 0.5814816355705261),
 ('toyota', 0.5654916763305664),
 ('fiat', 0.5608090162277222),
 ('bp', 0.5421079397201538),
 ('mitsubishi', 0.5372275114059448),
 ('honda', 0.5250917673110962),
 ('cubatão', 0.5177768468856812),
 ('patente', 0.513604998588562)]

## Salvando modelos 

In [64]:
w2v_model.wv.save_word2vec_format('/kaggle/working/model_cbow.txt', binary=False)
w2v_model_sg.wv.save_word2vec_format('/kaggle/working/model_sg.txt', binary=False)

# Testando com os modelos da part1

In [79]:
from gensim.models import KeyedVectors

w2v_model_cbow = KeyedVectors.load_word2vec_format('/kaggle/working/model_cbow.txt')
w2v_model_sg = KeyedVectors.load_word2vec_format('/kaggle/working/model_sg.txt')

nlp = spacy.load('pt_core_news_sm', disable=['paser', 'ner', 
                                             'tagger','textcat'])

def tokenizador(text):
    doc = nlp(text)
    tokens_validos = []
    for token in doc:
        valido = not token.is_stop and token.is_alpha
        if valido:
            tokens_validos.append(token.text.lower())

    return tokens_validos

def vetores_por_soma(palavras, model):
    vetor_resultante = np.zeros((1,300))
    for palavra in palavras:
        try:
            vetor_resultante += model.get_vector(palavra)
        except KeyError:
            pass
        
    return vetor_resultante

def matriz_vetores(texts, model):
    x = len(texts)
    y = 300
    matriz = np.zeros((x,y))
    
    for i in range(x):
        palavras = tokenizador(texts.iloc[i])
        matriz[i] = vetores_por_soma(palavras, model)
    
    return matriz

matriz_vet_train_cbow = matriz_vetores(train.title, w2v_model_cbow)
matriz_vet_test_cbow = matriz_vetores(test.title, w2v_model_cbow)
matriz_vet_train_cbow.shape, matriz_vet_test_cbow.shape

((90000, 300), (20513, 300))

In [80]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

def classification(model, x_train, y_train, x_test, y_test):
    lr = LogisticRegression(max_iter = 800)
    lr.fit(x_train, y_train)
    y_preds = lr.predict(x_test)
    resultado = classification_report(y_test, y_preds)
    print(resultado)

    return lr

lr_cbow = classification(w2v_model_cbow,
                        matriz_vet_train_cbow,
                        train.category,
                        matriz_vet_test_cbow,
                        test.category)

              precision    recall  f1-score   support

     colunas       0.81      0.71      0.76      6103
   cotidiano       0.63      0.80      0.71      1698
     esporte       0.93      0.87      0.90      4663
   ilustrada       0.13      0.85      0.22       131
     mercado       0.84      0.78      0.81      5867
       mundo       0.75      0.83      0.79      2051

    accuracy                           0.79     20513
   macro avg       0.68      0.81      0.70     20513
weighted avg       0.82      0.79      0.80     20513



In [86]:
matriz_vet_train_sg = matriz_vetores(train.title, w2v_model_sg)
matriz_vet_test_sg = matriz_vetores(test.title, w2v_model_sg)

lr_sg = classification(w2v_model_sg,
                        matriz_vet_train_sg,
                        train.category,
                        matriz_vet_test_sg,
                        test.category)

              precision    recall  f1-score   support

     colunas       0.81      0.71      0.76      6103
   cotidiano       0.63      0.79      0.71      1698
     esporte       0.93      0.87      0.90      4663
   ilustrada       0.14      0.85      0.23       131
     mercado       0.84      0.79      0.81      5867
       mundo       0.74      0.82      0.78      2051

    accuracy                           0.79     20513
   macro avg       0.68      0.81      0.70     20513
weighted avg       0.82      0.79      0.80     20513



In [88]:
import pickle

with open('/kaggle/working/lr_cbow.pkl', 'wb') as f:
    pickle.dump(lr_cbow, f)

with open('/kaggle/working/lr_sg.pkl', 'wb') as f:
    pickle.dump(lr_sg, f)

In [90]:
with open('/kaggle/working/lr_sg.pkl', 'rb') as f:
    best_model = pickle.load(f)

In [102]:
#Teste em produção
titulo = 'Corinthians eh o melhor time do mundo'
titulo_tokens = tokenizador(titulo)
titulo_vetor = vetores_por_soma(titulo_tokens, w2v_model_sg)
titulo_categoria = best_model.predict(titulo_vetor)
output = titulo_categoria[0].capitalize()
print(f'Categoria do título: {output}')

Categoria do título: Esporte


# Pontos importantes projeto

* O que é o Spacy;
* Como o Spacy se organiza;
* O que é a estrutura de dados DOC;
* Como instalar e usar os dados modelos linguísticos em português do Spacy.
* A tratar os dados para treinar um modelo Word2Vec;
* Como usar o Doc dos Spacy para ajudar no pré-processamento dos dados;
* Como paralelizar o pré-processamento com o Spacy (nlp.pipe).
* O que são Hiperparâmetros;
* Como os alguns Hiperparâmetros influenciam no modelo;
* A criar o vocabulário para treinamento do modelo Word2Vec.
* Como treinar seu modelo Word2Vec;
* A fazer uma avaliação qualitativa dos modelos;
* O workflow de treinamento do seu modelo Word2Vec.
* Como otimizar o tempo de pré-processamento desabilitando notações e parser na criação do objeto Doc do Spacy;
* Como criar as funções para o desenvolvimento do modelo de classificação;
* E a preparar os dados para criação os dados para a classificador.
* Como treinar o classificador de títulos;
* A salvar seu modelo de classificação em um arquivo pickle;